In [1]:
%load_ext autoreload
%autoreload 2

# B5 — Maintenance prédictive : chargement du Gold dataset

Chargement de `gold_dataset_20260622-080603.parquet` (dossier `indusense/`, en dehors de ce projet).

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

GOLD_FILE = Path('../indusense/gold_dataset_20260622-080603.parquet')

In [3]:
gold = pd.read_parquet(GOLD_FILE)

print('Shape :', gold.shape)
gold.head()

Shape : (134280, 100)


,machine_id_std,window_start,temp_mean_1h,temp_max_1h,pressure_mean_1h,pressure_max_1h,voltage_mean_1h,voltage_max_1h,rotation_mean_1h,rotation_max_1h,pieces_produced_sum_1h,window_end,temp_mean_6h,temp_max_6h,temp_std_6h,pressure_mean_6h,pressure_max_6h,pressure_std_6h,voltage_mean_6h,voltage_max_6h,...,type_surchauffe_count_prev_24h,type_baisse_pression_count_prev_24h,type_vibration_count_prev_24h,type_bruit_mecanique_count_prev_24h,type_surconsommation_count_prev_24h,type_blocage_mecanique_count_prev_24h,type_alarme_capteur_count_prev_24h,type_arret_urgence_count_prev_24h,type_defaut_qualite_count_prev_24h,days_since_last_maintenance,maintenance_count_prev_30d,future_incident_count_6h,label_failure_next_6h,future_incident_count_12h,label_failure_next_12h,future_incident_count_24h,label_failure_next_24h,future_incident_count_48h,label_failure_next_48h,split_set
0,MACH-01,2025-06-01 00:00:00,46.340,46.340,198.203,198.203,227.568,227.568,1541.787,1541.787,4,2025-06-01 01:00:00,46.3400,46.340,NaN,198.2030,198.203,NaN,227.568000,227.568,...,0,0,0,0,0,0,0,0,0,NaN,0,0.0,False,0.0,False,0.0,False,0.0,False,train
1,MACH-01,2025-06-01 01:00:00,48.762,48.762,198.295,198.295,227.480,227.480,1537.860,1537.860,4,2025-06-01 02:00:00,47.5510,48.762,1.712613,198.2490,198.295,0.065054,227.524000,227.568,...,0,0,0,0,0,0,0,0,0,NaN,0,0.0,False,0.0,False,0.0,False,0.0,False,train
2,MACH-01,2025-06-01 02:00:00,51.352,51.352,199.545,199.545,228.680,228.680,1584.660,1584.660,13,2025-06-01 03:00:00,48.8180,51.352,2.506469,198.6810,199.545,0.749659,227.909333,228.680,...,0,0,0,0,0,0,0,0,0,NaN,0,0.0,False,0.0,False,0.0,False,0.0,False,train
3,MACH-01,2025-06-01 03:00:00,49.512,49.512,201.641,201.641,228.440,228.440,1588.960,1588.960,10,2025-06-01 04:00:00,48.9915,51.352,2.075733,199.4210,201.641,1.601580,228.042000,228.680,...,0,0,0,0,0,0,0,0,0,NaN,0,0.0,False,0.0,False,0.0,False,0.0,False,train
4,MACH-01,2025-06-01 04:00:00,51.982,51.982,200.157,200.157,227.840,227.840,1548.660,1548.660,6,2025-06-01 05:00:00,49.5896,51.982,2.240562,199.5682,201.641,1.425529,228.001600,228.680,...,0,0,0,0,0,0,0,0,0,NaN,0,0.0,False,0.0,False,0.0,False,0.0,False,train


In [4]:
gold = gold.sort_values(['machine_id_std', 'window_start']).reset_index(drop=True)

gold[['machine_id_std', 'window_start']].head()

,machine_id_std,window_start
0,MACH-01,2025-06-01 00:00:00
1,MACH-01,2025-06-01 01:00:00
2,MACH-01,2025-06-01 02:00:00
3,MACH-01,2025-06-01 03:00:00
4,MACH-01,2025-06-01 04:00:00


- Choisir un **horizon** (ex. `label_failure_next_24h`) et construire la cible `y` (0/1).

On construit le "vecteur" des valeurs à prédire :

In [5]:
y = gold['label_failure_next_24h'].copy()

y.value_counts(normalize=True)

label_failure_next_24h
False    0.832082
True     0.167918
Name: proportion, dtype: float64

In [6]:
dropped_cols = [c for c in gold.columns if c.startswith('future_') or '_next_' in c]
dropped_cols += ['window_start', 'window_end']
gold = gold.drop(columns=dropped_cols)

print(f'Colonnes supprimées ({len(dropped_cols)}) :', dropped_cols)
print('Shape :', gold.shape)

Colonnes supprimées (10) : ['future_incident_count_6h', 'label_failure_next_6h', 'future_incident_count_12h', 'label_failure_next_12h', 'future_incident_count_24h', 'label_failure_next_24h', 'future_incident_count_48h', 'label_failure_next_48h', 'window_start', 'window_end']
Shape : (134280, 90)


In [7]:
gold = gold.iloc[1:].reset_index(drop=True)
y = y.iloc[1:].reset_index(drop=True)

print('Shape :', gold.shape, '| y :', y.shape)
gold.head()

Shape : (134279, 90) | y : (134279,)


,machine_id_std,temp_mean_1h,temp_max_1h,pressure_mean_1h,pressure_max_1h,voltage_mean_1h,voltage_max_1h,rotation_mean_1h,rotation_max_1h,pieces_produced_sum_1h,temp_mean_6h,temp_max_6h,temp_std_6h,pressure_mean_6h,pressure_max_6h,pressure_std_6h,voltage_mean_6h,voltage_max_6h,voltage_std_6h,rotation_mean_6h,...,type_baisse_pression,type_vibration,type_bruit_mecanique,type_surconsommation,type_blocage_mecanique,type_alarme_capteur,type_arret_urgence,type_defaut_qualite,type_surchauffe_count_prev_24h,type_baisse_pression_count_prev_24h,type_vibration_count_prev_24h,type_bruit_mecanique_count_prev_24h,type_surconsommation_count_prev_24h,type_blocage_mecanique_count_prev_24h,type_alarme_capteur_count_prev_24h,type_arret_urgence_count_prev_24h,type_defaut_qualite_count_prev_24h,days_since_last_maintenance,maintenance_count_prev_30d,split_set
0,MACH-01,48.762,48.762,198.295,198.295,227.48,227.48,1537.86,1537.86,4,47.5510,48.762,1.712613,198.249000,198.295,0.065054,227.524000,227.568,0.062225,1539.823500,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,train
1,MACH-01,51.352,51.352,199.545,199.545,228.68,228.68,1584.66,1584.66,13,48.8180,51.352,2.506469,198.681000,199.545,0.749659,227.909333,228.680,0.668866,1554.769000,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,train
2,MACH-01,49.512,49.512,201.641,201.641,228.44,228.44,1588.96,1588.96,10,48.9915,51.352,2.075733,199.421000,201.641,1.601580,228.042000,228.680,0.607170,1563.316750,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,train
3,MACH-01,51.982,51.982,200.157,200.157,227.84,227.84,1548.66,1548.66,6,49.5896,51.982,2.240562,199.568200,201.641,1.425529,228.001600,228.680,0.533529,1560.385400,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,train
4,MACH-01,50.402,50.402,199.914,199.914,227.50,227.50,1554.76,1554.76,4,49.7250,51.982,2.031279,199.625833,201.641,1.282824,227.918000,228.680,0.519284,1559.447833,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,train


- Utiliser le **split temporel fourni** (`split_set` : train < validation < test) — **pas** de split aléatoire.

On sépare `X` (features) et `y` en train / validation / test à partir de la colonne `split_set` :

In [8]:
X = gold.drop(columns='split_set')

train_mask = gold['split_set'] == 'train'
val_mask = gold['split_set'] == 'validation'
test_mask = gold['split_set'] == 'test'

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

for name, split_X in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    print(f'{name:<12} {split_X.shape[0]:>7} lignes')

train          93989 lignes
validation     20145 lignes
test           20145 lignes


- Imputer les `NaN` (médiane) ; standardiser pour les modèles linéaires.

In [9]:
# on liste les colonnes avec des valeurs à NaN

nan_counts = X.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)

print(f'Colonnes avec des NaN ({len(nan_counts)}/{X.shape[1]}) :')
nan_counts

Colonnes avec des NaN (43/89) :


incident_max_severity_1h          133237
incident_max_severity_prev_24h    111711
hours_since_last_incident           4027
days_since_last_maintenance         3445
rotation_trend_6h                   1722
rotation_delta_3h                   1360
rotation_delta_1h                   1085
rotation_max_1h                      948
rotation_mean_1h                     948
rotation_std_6h                      576
rotation_mean_6h                     350
rotation_max_6h                      350
pressure_trend_6h                    115
temp_trend_6h                         99
voltage_trend_6h                      89
rotation_std_12h                      82
pressure_delta_3h                     64
temp_delta_3h                         52
voltage_delta_3h                      44
pressure_delta_1h                     30
pressure_std_6h                       22
temp_delta_1h                         20
temp_zscore_24h                       19
rotation_max_12h                      18
rotation_mean_12

On impute donc avec la médiane :
hours_since_last_incident           4027
days_since_last_maintenance         3445
rotation_trend_6h                   1722
rotation_delta_3h                   1360
rotation_delta_1h                   1085
rotation_max_1h                      948
rotation_mean_1h                     948
rotation_std_6h                      576
rotation_mean_6h                     350
rotation_max_6h                      350
pressure_trend_6h                    115
temp_trend_6h                         99
voltage_trend_6h                      89
rotation_std_12h                      82
pressure_delta_3h                     64
temp_delta_3h                         52
voltage_delta_3h                      44
pressure_delta_1h                     30
pressure_std_6h                       22
temp_delta_1h                         20
temp_zscore_24h                       19
rotation_max_12h                      18
rotation_mean_12h                     18
temp_std_6h                           16
pressure_zscore_machine               14
voltage_delta_1h                      14
pressure_std_24h                      14
rotation_std_24h                      14
voltage_std_24h                       14
temp_std_24h                          14
voltage_std_12h                       14
pressure_std_12h                      14
temp_std_12h                          14
voltage_std_6h                        14
pressure_max_1h                       14
pressure_mean_1h                      14
temp_max_1h                            5
temp_zscore_machine                    5
temp_mean_1h                           5
pressure_max_6h                        4
pressure_mean_6h                       4

et avec 0 :
incident_max_severity_1h          133237
incident_max_severity_prev_24h    111711


In [10]:
# imputation pour chaque set

zero_cols = ['incident_max_severity_1h', 'incident_max_severity_prev_24h']
median_cols = [c for c in nan_counts.index if c not in zero_cols]

medians = X_train[median_cols].median()

for split in (X_train, X_val, X_test):
    split[median_cols] = split[median_cols].fillna(medians)
    split[zero_cols] = split[zero_cols].fillna(0)

print('NaN restants — train :', X_train.isna().sum().sum())
print('NaN restants — val   :', X_val.isna().sum().sum())
print('NaN restants — test  :', X_test.isna().sum().sum())

NaN restants — train : 0
NaN restants — val   : 0
NaN restants — test  : 0
